<a href="https://colab.research.google.com/github/OmNaidu123/MEDS/blob/main/MEDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
# Use the definitions from the two cells above (no need to save as files)

# Choose any binary message; with k=10 and n=3, we effectively store 30 base-2 digits.
secret_bits = "1234"  # any length; it will be left-padded to 30 bits

# Send
session_id, mem_buckets = embed_secret(secret_bits, KEY, use_memory_fallback=False)  # set True to test without DBs

[Sender] Embedded session_id=SID-60a314b1 across n=3 clouds, k=10 lists, base B=2.


In [48]:
# Receive (and auto-delete)
recovered = extract_secret_and_cleanup(KEY, session_id, memory_buckets=None)  # pass mem_buckets if you used fallback
print("Recovered:", recovered)

[Receiver] Reconstructed integer=67, secret_bits='1000011'. Deleted 30 docs.
Recovered: 1000011


In [42]:
# --- Sender: embed a secret across 3 MongoDB accounts by writing only "filenames" (strings) ---
# Run this in Google Colab. First time:  !pip -q install pymongo

from dataclasses import dataclass
from typing import List, Dict, Any, Optional
import uuid

try:
    from pymongo import MongoClient
except Exception:
    MongoClient = None  # If pymongo not installed yet, you'll get the assert message when used.


# ==========================
# CONFIG / SHARED KEY
# ==========================
# Base B=2; k=10 lists; n=3 clouds (accounts). Both sender and receiver MUST use the same KEY.
KEY = {
    "B": 2,  # base fixed to 2
    "clouds": [
        {"name": "c0", "type": "mongo", "uri": "<DB_URI>", "db": "stego_db", "collection": "files"},
        {"name": "c1", "type": "mongo", "uri": "<DB_URI>", "db": "stego_db", "collection": "files"},
        {"name": "c2", "type": "mongo", "uri": "<DB_URI>", "db": "stego_db", "collection": "files"},
    ],
    # k = 10 lists; each list length is B=2 (index 0 or 1).
    # Index 0 => first filename (e.g., "hello.pdf"), Index 1 => second filename (e.g., "world.pdf")
    "lists": [
        ["hello.pdf", "world.pdf"],             # L(0)
        ["alpha.docx", "beta.docx"],            # L(1)
        ["notes.txt", "final.txt"],             # L(2)
        ["plan.xlsx", "budget.xlsx"],           # L(3)
        ["intro.pptx", "summary.pptx"],         # L(4)
        ["task.csv", "result.csv"],             # L(5)
        ["image.png", "diagram.png"],           # L(6)
        ["draft.md", "report.md"],              # L(7)
        ["readme.md", "license.md"],            # L(8)
        ["data.json", "config.json"],           # L(9)
    ],
    "namespace": "project_demo_b2_k10"  # tag so you can run multiple demos without collisions
}


# ==========================
# STORAGE ABSTRACTION
# ==========================
class CloudStore:
    def put(self, doc: Dict[str, Any]) -> None:
        raise NotImplementedError

@dataclass
class MongoStore(CloudStore):
    uri: str
    db_name: str
    collection: str
    def __post_init__(self):
        assert MongoClient is not None, "Install pymongo first:  !pip -q install pymongo"
        self.client = MongoClient(self.uri, tlsAllowInvalidCertificates=True)
        self.col = self.client[self.db_name][self.collection]
    def put(self, doc: Dict[str, Any]) -> None:
        self.col.insert_one(doc)

# Optional in-memory fallback for quick dry runs (set use_memory_fallback=True)
class MemoryStore(CloudStore):
    def __init__(self, bucket: List[Dict[str, Any]]):
        self.bucket = bucket
    def put(self, doc: Dict[str, Any]) -> None:
        self.bucket.append(doc)


# ==========================
# HELPERS
# ==========================
def int_to_base_digits(x: int, base: int) -> List[int]:
    if x == 0:
        return [0]
    digs = []
    while x > 0:
        digs.append(x % base)
        x //= base
    return list(reversed(digs))

def pad_left(values: List[int], total_len: int, pad_val: int = 0) -> List[int]:
    return [pad_val] * max(0, (total_len - len(values))) + values



# ==========================
# EMBEDDING (SENDER)
# ==========================
def embed_secret(secret_bits: str, key: Dict[str, Any], session_id: Optional[str] = None,
                 use_memory_fallback: bool = False):
    """
    secret_bits: binary string, e.g., '1011001'.
    Strategy:
      1) Convert bits -> integer -> base-B digits (here B=2, so it's essentially the same bits).
      2) Pad left to k*n digits (k rows, n clouds).
      3) Split into k blocks of n digits.
      4) For each block i and cloud j, choose filename lists[i][value] and write a small doc to cloud j.
    """
    if session_id is None:
        session_id = f"SID-{uuid.uuid4().hex[:8]}"

    B = key["B"]
    lists = key["lists"]
    k = len(lists)                  # number of rows (lists)
    n = len(key["clouds"])          # number of clouds (columns)
    assert all(len(L) == B for L in lists), "Each list must have exactly B items (since base=2, each list has 2)."

    # Convert secret bits to integer, then to base-B digit list
    s_int = int(secret_bits) if len(secret_bits) > 0 else 0
    base_digits = int_to_base_digits(s_int, B)  # with B=2 this mirrors the bit string

    # We’ll write exactly k*n digits. If fewer, left-pad with zeros (leading zeros are fine).
    needed_len = k * n
    digits = pad_left(base_digits, needed_len, 0)

    # Build blocks: k blocks, each of length n (one digit per cloud)
    blocks = [digits[i * n:(i + 1) * n] for i in range(k)]

    # Prepare cloud stores
    stores: List[CloudStore] = []
    memory_buckets = [[] for _ in range(n)]
    for idx, c in enumerate(key["clouds"]):
        if not use_memory_fallback and c["type"] == "mongo" and "<MONGODB_URI_" not in c["uri"]:
            stores.append(MongoStore(c["uri"], c["db"], c["collection"]))
        else:
            stores.append(MemoryStore(memory_buckets[idx]))

    namespace = key.get("namespace", "default_ns")

    # Write: for each row i and cloud j, choose filename by the digit value (0 or 1)
    for i, block in enumerate(blocks):
        for j, v in enumerate(block):
            filename = lists[i][v]              # v ∈ {0,1}
            doc = {
                "ns": namespace,
                "session_id": session_id,
                "list_idx": i,                  # row index
                "cloud_ordinal": j,             # column index
                "value": v,                     # optional: stored for debugging
                "filename": filename,           # the “pointer” string
            }
            stores[j].put(doc)

    print(f"[Sender] Embedded session_id={session_id} across n={n} clouds, k={k} lists, base B={B}.")
    if use_memory_fallback:
        print("[Sender] (In-memory fallback used; nothing written to MongoDB.)")
    return session_id, memory_buckets


# ==========================
# QUICK DEMO (optional)
# ==========================
if __name__ == "__main__":
    # Example: 30 bits are used (k=10, n=3) — we'll left-pad if you give fewer
    example_bits = "101101"  # put anything here
    sid, _ = embed_secret(example_bits, KEY, use_memory_fallback=True)
    print("Session ID:", sid)


[Sender] Embedded session_id=SID-efb55e5d across n=3 clouds, k=10 lists, base B=2.
[Sender] (In-memory fallback used; nothing written to MongoDB.)
Session ID: SID-efb55e5d


In [41]:
# --- Receiver: reconstruct secret from the 3 MongoDB accounts, then DELETE used records ---
# Run this in Google Colab. First time:  !pip -q install pymongo

from dataclasses import dataclass
from typing import List, Dict, Any, Optional

try:
    from pymongo import MongoClient
except Exception:
    MongoClient = None


# ==========================
# CONFIG / SHARED KEY
# ==========================
# Must be IDENTICAL to sender’s KEY.
KEY = {
    "B": 2,
    "clouds": [
        {"name": "c0", "type": "mongo", "uri": "<DB_URI>", "db": "stego_db", "collection": "files"},
        {"name": "c1", "type": "mongo", "uri": "<DB_URI>", "db": "stego_db", "collection": "files"},
        {"name": "c2", "type": "mongo", "uri": "<DB_URI>", "db": "stego_db", "collection": "files"},
    ],
    "lists": [
        ["hello.pdf", "world.pdf"],             # L(0)
        ["alpha.docx", "beta.docx"],            # L(1)
        ["notes.txt", "final.txt"],             # L(2)
        ["plan.xlsx", "budget.xlsx"],           # L(3)
        ["intro.pptx", "summary.pptx"],         # L(4)
        ["task.csv", "result.csv"],             # L(5)
        ["image.png", "diagram.png"],           # L(6)
        ["draft.md", "report.md"],              # L(7)
        ["readme.md", "license.md"],            # L(8)
        ["data.json", "config.json"],           # L(9)
    ],
    "namespace": "project_demo_b2_k10"
}


# ==========================
# STORAGE ABSTRACTION
# ==========================
class CloudStoreReader:
    def find(self, query: Dict[str, Any]) -> List[Dict[str, Any]]:
        raise NotImplementedError
    def delete_many(self, query: Dict[str, Any]) -> int:
        raise NotImplementedError

@dataclass
class MongoStoreReader(CloudStoreReader):
    uri: str
    db_name: str
    collection: str
    def __post_init__(self):
        assert MongoClient is not None, "Install pymongo first:  !pip -q install pymongo"
        self.client = MongoClient(self.uri, tlsAllowInvalidCertificates=True)
        self.col = self.client[self.db_name][self.collection]
    def find(self, query: Dict[str, Any]) -> List[Dict[str, Any]]:
        # Sort by list_idx for stability (not required, but neat)
        return list(self.col.find(query).sort([("list_idx", 1), ("cloud_ordinal", 1)]))
    def delete_many(self, query: Dict[str, Any]) -> int:
        return self.col.delete_many(query).deleted_count

# Optional in-memory fallback
class MemoryStoreReader(CloudStoreReader):
    def __init__(self, bucket: List[Dict[str, Any]]):
        self.bucket = bucket
    def find(self, query: Dict[str, Any]) -> List[Dict[str, Any]]:
        out = []
        for d in self.bucket:
            if all(d.get(k) == v for k, v in query.items()):
                out.append(d)
        # Stable ordering like Mongo sort:
        out.sort(key=lambda x: (x.get("list_idx", -1), x.get("cloud_ordinal", -1)))
        return out
    def delete_many(self, query: Dict[str, Any]) -> int:
        keep = []
        removed = 0
        for d in self.bucket:
            if all(d.get(k) == v for k, v in query.items()):
                removed += 1
            else:
                keep.append(d)
        self.bucket[:] = keep
        return removed


# ==========================
# HELPERS
# ==========================
def base_matrix_to_int(Mat: List[List[int]], B: int) -> int:
    """
    Interpret Mat so that bottom-right is the least-significant digit.
    Traversal order: i = k-1..0, j = n-1..0
    """
    if not Mat:
        return 0
    k = len(Mat)
    n = len(Mat[0])
    total = 0
    pow_val = 1
    for i in range(k - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            total += Mat[i][j] * pow_val
            pow_val *= B
    return total

def int_to_binary_str(x: int) -> str:
    return bin(x)[2:] if x > 0 else "0"


# ==========================
# EXTRACTION (RECEIVER)
# ==========================
def extract_secret_and_cleanup(key: Dict[str, Any], session_id: str,
                               memory_buckets: Optional[List[List[Dict[str, Any]]]] = None) -> str:
    """
    1) Read docs for this (ns, session_id) from each cloud.
    2) Build Mat[k][n] where Mat[i][j] is the index (0/1) found by mapping filename back to lists[i].
    3) Decode to integer then to binary.
    4) Delete all docs with (ns, session_id) from every cloud.
    """
    B = key["B"]
    lists = key["lists"]
    k = len(lists)
    n = len(key["clouds"])
    namespace = key.get("namespace", "default_ns")

    # Open readers
    readers: List[CloudStoreReader] = []
    for idx, c in enumerate(key["clouds"]):
        if memory_buckets is not None:
            readers.append(MemoryStoreReader(memory_buckets[idx]))
        else:
            assert "<MONGODB_URI_" not in c["uri"], "Fill in your real MongoDB URIs in KEY['clouds']."
            readers.append(MongoStoreReader(c["uri"], c["db"], c["collection"]))

    # Fetch per cloud
    per_cloud_docs: List[List[Dict[str, Any]]] = []
    for r in readers:
        per_cloud_docs.append(r.find({"ns": namespace, "session_id": session_id}))

    # Build matrix Mat[k][n], default 0 if anything missing (left-pad behavior mirrored)
    Mat = [[0 for _ in range(n)] for __ in range(k)]
    for j, docs in enumerate(per_cloud_docs):
        for d in docs:
            i = d.get("list_idx", None)
            if i is None or not (0 <= i < k):
                continue
            filename = d.get("filename", None)
            if filename is None:
                continue
            try:
                idx_in_list = lists[i].index(filename)  # 0 or 1
            except ValueError:
                # filename not part of L(i); ignore
                continue
            Mat[i][j] = idx_in_list

    # Decode
    m = base_matrix_to_int(Mat, B)
    secret_bits = int_to_binary_str(m)

    # Cleanup (delete all used docs for this session across all clouds)
    total_deleted = 0
    for r in readers:
        total_deleted += r.delete_many({"ns": namespace, "session_id": session_id})

    print(f"[Receiver] Reconstructed integer={m}, secret_bits='{secret_bits}'. Deleted {total_deleted} docs.")
    return secret_bits


# ==========================
# QUICK DEMO (optional)
# ==========================
if __name__ == "__main__":
    # Example usage with in-memory buckets (simulate clouds):
    # - You’d normally pass memory_buckets returned by sender.embed_secret(..., use_memory_fallback=True)
    print("Receiver demo: run the sender first and capture memory_buckets for a dry run.")


Receiver demo: run the sender first and capture memory_buckets for a dry run.


In [ ]:
pip install "pymongo[srv]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 76.1 MB/s eta 0:00:00
